<a href="https://colab.research.google.com/github/Psyche-1/goit-ds-hw/blob/main/Hw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
from scipy.stats import skew, kurtosis, entropy

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [6]:
base_path = Path('/content/drive/MyDrive/Colab Notebooks/Data/data')
categories = ['walking', 'stairs', 'running', 'idle']

data_frames = []

for category in categories:
    folder_path = base_path / category
    csv_files = folder_path.glob('*.csv')

    for file_path in csv_files:
        temp_df = pd.read_csv(file_path)
        temp_df['activity'] = category
        data_frames.append(temp_df)

df = pd.concat(data_frames, ignore_index=True)

In [7]:
df

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity
0,1.240197,-4.481945,-6.962338,walking
1,3.227384,-8.566454,1.029507,walking
2,-3.600879,-16.721106,-17.707516,walking
3,0.885855,-7.024587,0.981623,walking
4,-1.412579,-2.513912,5.635951,walking
...,...,...,...,...
193855,0.268151,0.086191,9.725247,idle
193856,0.368707,-0.004788,9.777920,idle
193857,0.411803,-0.057461,9.777920,idle
193858,0.469264,-0.076614,9.806650,idle


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193860 entries, 0 to 193859
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   accelerometer_X  193860 non-null  float64
 1   accelerometer_Y  193860 non-null  float64
 2   accelerometer_Z  193860 non-null  float64
 3   activity         193860 non-null  object 
dtypes: float64(3), object(1)
memory usage: 5.9+ MB


In [9]:
df.describe()

,accelerometer_X,accelerometer_Y,accelerometer_Z
count,193860.000000,193860.000000,193860.000000
mean,1.923550,1.598343,1.804896
std,8.404867,12.474041,7.191590
min,-39.188293,-39.188293,-39.188293
25%,-2.494758,-8.327033,-2.494758
50%,0.248997,-0.009577,0.905008
75%,4.668694,8.671799,7.187394
max,39.188293,39.188293,39.188293


In [10]:
df.duplicated().sum()

np.int64(180673)

In [11]:
df_cleaned = df.drop_duplicates()
df_cleaned

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity
0,1.240197,-4.481945,-6.962338,walking
1,3.227384,-8.566454,1.029507,walking
2,-3.600879,-16.721106,-17.707516,walking
3,0.885855,-7.024587,0.981623,walking
4,-1.412579,-2.513912,5.635951,walking
...,...,...,...,...
191927,-1.599327,0.110133,8.700529,idle
193440,1.000776,4.616021,8.576031,idle
193441,0.718261,4.209007,8.446744,idle
193558,-0.177171,4.692636,8.394072,idle


In [72]:
WINDOW_SIZE = 30

axes = ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z']

def extract_features_from_window(window_df):
    features = {}

    features['activity'] = window_df['activity'].iloc[0]

    for axis in axes:
        signal = window_df[axis].to_numpy()

        features[f'{axis}_mean'] = np.mean(signal)
        features[f'{axis}_variance'] = np.var(signal)
        features[f'{axis}_std'] = np.std(signal)
        features[f'{axis}_median'] = np.median(signal)
        features[f'{axis}_min'] = np.min(signal)
        features[f'{axis}_max'] = np.max(signal)
        features[f'{axis}_range'] = features[f'{axis}_max'] - features[f'{axis}_min']
        features[f'{axis}_rms'] = np.sqrt(np.mean(signal**2))
        features[f'{axis}_idx_min'] = np.argmin(signal)
        features[f'{axis}_idx_max'] = np.argmax(signal)
        features[f'{axis}_power'] = np.mean(signal**2)
        features[f'{axis}_energy'] = np.sum(signal**2)
        features[f'{axis}_skewness'] = skew(signal) if np.std(signal) > 0 else 0
        features[f'{axis}_kurtosis'] = kurtosis(signal) if np.std(signal) > 0 else 0
        features[f'{axis}_iqr'] = np.percentile(signal, 75) - np.percentile(signal, 25)
        features[f'{axis}_mad'] = np.mean(np.abs(signal - np.mean(signal)))

        hist, _ = np.histogram(signal, bins=10, density=True)
        hist = hist[hist > 0]
        features[f'{axis}_entropy'] = entropy(hist) if len(hist) > 0 else 0

    acc_x, acc_y, acc_z = window_df[axes[0]], window_df[axes[1]], window_df[axes[2]]
    features['SMA'] = np.sum(np.abs(acc_x) + np.abs(acc_y) + np.abs(acc_z)) / len(window_df)

    features['corr_xy'] = np.corrcoef(acc_x, acc_y)[0, 1] if np.std(acc_x) > 0 and np.std(acc_y) > 0 else 0
    features['corr_xz'] = np.corrcoef(acc_x, acc_z)[0, 1] if np.std(acc_x) > 0 and np.std(acc_z) > 0 else 0
    features['corr_yz'] = np.corrcoef(acc_y, acc_z)[0, 1] if np.std(acc_y) > 0 and np.std(acc_z) > 0 else 0

    return features

df['activity_block'] = (df['activity'] != df['activity'].shift()).cumsum()

df['internal_index'] = df.groupby('activity_block').cumcount()
df['window_id'] = df['activity_block'].astype(str) + "_" + (df['internal_index'] // WINDOW_SIZE).astype(str)

feature_rows = []
for wid, group in df.groupby('window_id'):
    if len(group) == WINDOW_SIZE:
        feature_rows.append(extract_features_from_window(group))

df_features = pd.DataFrame(feature_rows)
df_features

,activity,accelerometer_X_mean,accelerometer_X_variance,accelerometer_X_std,accelerometer_X_median,accelerometer_X_min,accelerometer_X_max,accelerometer_X_range,accelerometer_X_rms,accelerometer_X_idx_min,...,accelerometer_Z_energy,accelerometer_Z_skewness,accelerometer_Z_kurtosis,accelerometer_Z_iqr,accelerometer_Z_mad,accelerometer_Z_entropy,SMA,corr_xy,corr_xz,corr_yz
0,walking,-0.140300,32.089108,5.664725,-0.670377,-21.485565,12.952631,34.438196,5.666462,7,...,1435.259652,-0.859128,0.247781,6.404490,4.927267,2.075344,19.005652,-0.553122,0.398718,0.208325
1,walking,-0.510923,8.141269,2.853291,-0.600944,-7.244854,4.721366,11.966220,2.898674,21,...,1653.315067,-0.348116,2.655634,6.540959,5.140543,1.549594,17.049270,0.064763,-0.095490,0.412003
2,walking,1.081541,20.633895,4.542455,0.780510,-6.670246,9.370906,16.041152,4.669435,21,...,847.725902,0.386946,0.041285,7.681796,4.254401,1.945733,16.925570,-0.451367,-0.295089,0.359570
3,walking,-3.058832,12.818368,3.580275,-2.346318,-13.962984,1.240197,15.203181,4.709015,29,...,530.924383,0.731853,5.522464,3.463213,2.641517,1.398671,15.339331,0.176185,0.245330,0.197147
4,walking,-1.259350,16.669648,4.082848,-1.551442,-9.323022,11.894394,21.217416,4.272659,4,...,1295.424946,0.456560,3.111362,5.310339,4.255826,1.471712,18.155710,0.062909,-0.196432,0.657348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6457,idle,0.399193,0.001383,0.037183,0.390255,0.330400,0.488417,0.158017,0.400921,24,...,2855.518490,0.248461,-1.126255,0.027533,0.014344,2.257279,10.192436,-0.338821,0.164114,-0.080443
6458,idle,0.244368,1.154142,1.074310,0.304064,-2.044648,4.797981,6.842629,1.101752,21,...,2076.128872,-0.201059,-0.376490,1.569399,0.882343,2.004770,13.254620,0.102023,-0.420381,-0.651297
6459,idle,0.216276,0.374391,0.611875,0.375889,-2.044648,1.225831,3.270479,0.648973,9,...,2441.282318,-1.224861,0.446798,1.363498,0.952169,1.418484,11.452584,-0.163819,-0.073688,-0.839075
6460,idle,-0.099279,0.000232,0.015240,-0.100556,-0.124498,-0.062249,0.062249,0.100442,9,...,2863.275915,0.012714,-0.746111,0.017957,0.010843,1.714088,10.088049,-0.132185,0.106564,-0.241086


In [73]:
feature_cols = [col for col in df_features.columns if col not in ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z', 'activity']]

X = df_features[feature_cols]
y = df_features['activity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [90]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)

In [91]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       1.00      0.94      0.97        33
     walking       0.99      1.00      1.00       370

    accuracy                           1.00      1293
   macro avg       1.00      0.98      0.99      1293
weighted avg       1.00      1.00      1.00      1293



In [82]:
svm_model = SVC(kernel='rbf', C=10.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)

In [83]:
print(classification_report(y_test, svm_pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       208
     running       1.00      1.00      1.00       682
      stairs       1.00      1.00      1.00        33
     walking       1.00      1.00      1.00       370

    accuracy                           1.00      1293
   macro avg       1.00      1.00      1.00      1293
weighted avg       1.00      1.00      1.00      1293



Модель RandomForestClassifier відпрацювала майже ідеально.

Модель SVM відпрацювала ідеально.